### <span style="color:lightgreen"> Hyperparameter Tuning for RNN Models </span>

In [8]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from scikeras.wrappers import KerasRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Load Data
DATA_PATH = "restaurant_footfall.csv"
data = pd.read_csv(DATA_PATH, parse_dates=["Date"], index_col="Date")

# Normalize Data
scaler = MinMaxScaler()
data_scaled = scaler.fit_transform(data)

# Function to create sequences
def create_sequences(data, seq_length):
    X, y = [], []
    for i in range(len(data) - seq_length):
        X.append(data[i:i+seq_length])
        y.append(data[i+seq_length])
    return np.array(X), np.array(y)

SEQ_LENGTH = 10  # Sequence length
X, y = create_sequences(data_scaled, SEQ_LENGTH)

# Train-test split
split_idx = int(len(X) * 0.8)
X_train, X_test, y_train, y_test = X[:split_idx], X[split_idx:], y[:split_idx], y[split_idx:]

# ✅ Define model function (ensure arguments match param_grid)
def build_rnn_model(num_neurons=50, dropout_rate=0.2, learning_rate=0.001):
    model = Sequential([
        SimpleRNN(num_neurons, activation='tanh', return_sequences=False, input_shape=(SEQ_LENGTH, 1)),
        Dropout(dropout_rate),
        Dense(1)
    ])
    model.compile(optimizer=Adam(learning_rate=learning_rate), loss="mse")
    return model

# ✅ Wrap model with KerasRegressor (pass hyperparameters here)
model = KerasRegressor(
    model=build_rnn_model,
    num_neurons=50,  
    dropout_rate=0.2,
    learning_rate=0.001,
    verbose=0
)

# ✅ Define correct hyperparameter grid (must match build_rnn_model args)
param_grid = {
    'num_neurons': [32, 64, 128],  
    'dropout_rate': [0.2, 0.3, 0.5],  
    'learning_rate': [0.001, 0.005, 0.01],  
    'batch_size': [16, 32],  
    'epochs': [50, 100]  
}

# Perform Grid Search
grid_search = GridSearchCV(estimator=model, param_grid=param_grid, cv=3, n_jobs=-1)
grid_search.fit(X_train, y_train)

# Best Parameters
best_params = grid_search.best_params_
print(f"Best Hyperparameters: {best_params}")

# Train final model with best parameters
best_model = build_rnn_model(learning_rate=best_params['learning_rate'],
                             num_neurons=best_params['num_neurons'],
                             dropout_rate=best_params['dropout_rate'])

history = best_model.fit(X_train, y_train, epochs=best_params['epochs'], batch_size=best_params['batch_size'], validation_data=(X_test, y_test), verbose=1)

# Step 6: Evaluate the Model
y_pred = best_model.predict(X_test)
y_test_actual = scaler.inverse_transform(y_test.reshape(-1, 1))
y_pred_actual = scaler.inverse_transform(y_pred)

mae = mean_absolute_error(y_test_actual, y_pred_actual)
mse = mean_squared_error(y_test_actual, y_pred_actual)
print(f"Mean Absolute Error (MAE): {mae:.2f}")
print(f"Mean Squared Error (MSE): {mse:.2f}")


# Step 7: Predict Future Footfall
def predict_future(model, last_sequence, days_to_predict):
    predictions = []
    current_seq = last_sequence
    for _ in range(days_to_predict):
        next_day_pred = model.predict(current_seq.reshape(1, SEQ_LENGTH, 1))
        predictions.append(next_day_pred[0, 0])
        current_seq = np.append(current_seq[1:], [[next_day_pred[0, 0]]], axis=0)
    return np.array(predictions)

last_seq = X_test[-1]
future_predictions = predict_future(best_model, last_seq, 10)
future_predictions_actual = scaler.inverse_transform(future_predictions.reshape(-1, 1))
print("Future Predictions:", [int(pred) for pred in future_predictions_actual.flatten()])



/opt/anaconda3/envs/myenv/lib/python3.11/site-packages/keras/src/layers/rnn/rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
/opt/anaconda3/envs/myenv/lib/python3.11/site-packages/keras/src/layers/rnn/rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
/opt/anaconda3/envs/myenv/lib/python3.11/site-packages/keras/src/layers/rnn/rnn.py:200: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
/opt/anaconda3/envs/myenv/lib/python3.11/site-packages/keras/src/layers/rnn/rnn.py:200: UserWarni

Best Hyperparameters: {'batch_size': 32, 'dropout_rate': 0.5, 'epochs': 100, 'learning_rate': 0.005, 'num_neurons': 64}
Epoch 1/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - loss: 0.3143 - val_loss: 0.0163
Epoch 2/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0885 - val_loss: 0.0173
Epoch 3/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0344 - val_loss: 0.0162
Epoch 4/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0407 - val_loss: 0.0155
Epoch 5/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0262 - val_loss: 0.0146
Epoch 6/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0312 - val_loss: 0.0140
Epoch 7/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0274 - val_loss: 0.0133
Epoch 8/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0231 - val_loss: 0.0145
Epoch 9/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0230 - val_loss: 0.0127
Epoch 10/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.0219 - val_loss: 0.0127
Epoch 11/100
9/9 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/ste